In [1]:
import os
import json
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from scipy.optimize import curve_fit
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

In [3]:
trf_locations = sorted(
    glob.glob(
        os.path.join('experiments', '*', 'trf')
    ), key = lambda x: int(x.split(os.path.sep)[1])
)
lstm_locations = sorted(
    glob.glob(
        os.path.join('experiments', '*', 'lstm')
    ), key = lambda x: int(x.split(os.path.sep)[1])
)

In [4]:
markers = [
    'o', '*', 'x', '+', 's', 'D', 'p', 'h', 'v', '^'
]

In [5]:
def plot_run(locations, vocab_size):
    locations_copy = []
    for location in locations:
        data = json.load(open(os.path.join(location, 'hparams.json'), 'r', encoding='utf-8'))
        if data['grammar_num_symbols'] == vocab_size:
            if data['grammar_type'] == 'pcfg':
                if data['grammar_seed'] == 0:
                    locations_copy.append(location)
    
    fig, axs = plt.subplots(
        nrows=len(locations_copy),
        ncols=1,
        sharex='all',
        # sharey='all',
        figsize=(10, 3 * len(locations_copy))  # Add height: 3 inches per subplot
    )
    
    for j, location in enumerate(locations_copy):
        data = json.load(open(os.path.join(location, 'hparams.json'), 'r', encoding='utf-8'))
        
        if data['grammar_type'] == 'pcfg':
            title = f"num_states={data['grammar_formalism_arg']}"
        
        length_wise_metrics = pd.read_csv(
            os.path.join(location, 'length_wise_metrics.tsv'), sep='\t'
        )
        
        length_wise_metrics = length_wise_metrics[length_wise_metrics['step'] <= 10_000]
        
        for i in range(10):
            subset = length_wise_metrics[length_wise_metrics['seq_len'] == (i+1)]
            axs[j].plot(
                subset['step'],
                subset['rho'],
                c=colors[i],
                # s=30,
                label=f'SLAC@{i+1}',
                marker=markers[i],
                alpha=0.5
            )
        axs[j].grid()
        axs[j].legend(loc='upper right')
        axs[j].set_ylabel(title)
        
    font = {
        'size': 14,
    }
    
    fig.suptitle(f'SLAC@k over training steps PCFG(seed=0, num_symbols={vocab_size})', y=0.98)
    fig.text(0.5, 0.03, 'Training steps', ha='center', fontdict=font)
    fig.text(0.04, 0.5, 'SLAC@k over evaluation set', va='center', rotation='vertical', fontdict=font)
    plt.tight_layout(rect=[0.06, 0.04, 1, 0.99])
    # plt.savefig(f'figs/runs_{vocab_size}.pdf')
    plt.show()

In [6]:
# plot_run(trf_locations, 1000)

In [7]:
# plot_run(trf_locations, 5000)

In [8]:
lstm_or_trf = []
seeds = []
formalisms = []
entropy = []
train_data_ee = []
val_data_ee = []
num_symbols = []
num_states = []
max_rhos = []
max_rho_num_steps = []

k = 10

for loc in trf_locations:
    data = json.load(open(os.path.join(loc, 'hparams.json')))
    length_wise_metrics = pd.read_csv(os.path.join(loc, 'length_wise_metrics.tsv'), sep='\t')
    subset = length_wise_metrics[length_wise_metrics['seq_len'] <= k]
    means = subset.groupby('step').mean()
    means['step'] = means.index
    max_rho_at_k = means.max()['rho']
    max_rhos.append(max_rho_at_k)
    max_rho_num_steps.append(means[means['rho'] == max_rho_at_k]['step'].item())
    lstm_or_trf.append('trf')
    entropy.append(data['grammar_actual_entropy'])
    train_data_ee.append(data['train_data_ee'])
    val_data_ee.append(data['val_data_ee'])
    num_symbols.append(data['grammar_num_symbols'])
    num_states.append(data['grammar_formalism_arg'])
    seeds.append(data['grammar_seed'])
    formalisms.append(data['grammar_type'])
    
for loc in lstm_locations:
    data = json.load(open(os.path.join(loc, 'hparams.json')))
    length_wise_metrics = pd.read_csv(os.path.join(loc, 'length_wise_metrics.tsv'), sep='\t')
    subset = length_wise_metrics[length_wise_metrics['seq_len'] <= k]
    means = subset.groupby('step').mean()
    means['step'] = means.index
    max_rho_at_k = means.max()['rho']
    max_rhos.append(max_rho_at_k)
    max_rho_num_steps.append(means[means['rho'] == max_rho_at_k]['step'].item())
    lstm_or_trf.append('lstm')
    entropy.append(data['grammar_actual_entropy'])
    train_data_ee.append(data['train_data_ee'])
    val_data_ee.append(data['val_data_ee'])
    num_symbols.append(data['grammar_num_symbols'])
    num_states.append(data['grammar_formalism_arg'])
    seeds.append(data['grammar_seed'])
    formalisms.append(data['grammar_type'])
    
df = pd.DataFrame({
    'lstm_or_trf': lstm_or_trf,
    'formalism': formalisms,
    'seed': seeds,
    'entropy': entropy,
    'train_data_ee': train_data_ee,
    'val_data_ee': val_data_ee,
    'num_symbols': num_symbols,
    'num_states': num_states,
    'max_rho': max_rhos,
    'max_rho_num_steps': max_rho_num_steps
})

In [9]:
df['num_states'] = np.log2(df['num_states']).astype(int)
df['seed'] = df['seed'].astype(str)
df['ee'] = (df['train_data_ee'] + df['val_data_ee']) / 2

# outliers
df = df[df['entropy'] < 20]

In [10]:
df = df[df['formalism'] == 'pcfg']
df = df[df['entropy'] > 0]

In [11]:
print(df)

    lstm_or_trf formalism seed    entropy  train_data_ee  val_data_ee  \
37          trf      pcfg    0   7.981645            0.0          0.0   
39          trf      pcfg    1   6.583272            0.0          0.0   
41          trf      pcfg    0   6.414402            0.0          0.0   
42          trf      pcfg    0   7.023314            0.0          0.0   
43          trf      pcfg    0  10.671010            0.0          0.0   
45          trf      pcfg    1   8.043761            0.0          0.0   
46          trf      pcfg    0   8.178726            0.0          0.0   
48          trf      pcfg    0   8.808687            0.0          0.0   
49          trf      pcfg    1   7.655004            0.0          0.0   
50          trf      pcfg    0   8.013559            0.0          0.0   
51          trf      pcfg    1  11.527168            0.0          0.0   
52          trf      pcfg    0  12.263765            0.0          0.0   
53          trf      pcfg    1   8.229737          

In [12]:
fig = px.scatter(
    df,
    'entropy',
    'max_rho',
    symbol='lstm_or_trf',
    size='num_symbols',
    title='SLAC@(1,10) vs. Entropy (size indicates # symbols)',
    color='num_states',
    labels={
        'entropy': 'Entropy (nats)',
        'max_rho': 'SLAC@(1,10)',
        'lstm_or_trf': 'LSTM/TRF',
        'num_symbols': '# Symbols',
        'num_states': 'log_2(# States)'
    },
    opacity=0.5
).update_layout(
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.8,
        xanchor='left',
        x=1.2,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='rgba(0, 0, 0, 0.2)',
        borderwidth=1
    ),
    margin=dict(r=200),
    hovermode='closest',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=20)
)
fig.show()
fig.write_image('figs/slac_pcfg.pdf', format='pdf', width=1000)

In [13]:
def f(x, a, b, c, d):
    return a * np.log(b * x + c) + d

fig = px.scatter(
    df,
    'entropy',
    'ee',
    size='num_symbols',
    title='Excess entropy vs. Entropy',
    color='num_states',
    labels={
        'ee': 'Excess entropy',
        'entropy': 'Entropy',
        'max_rho': 'SLAC@(1,10)',
        'num_symbols': '# Symbols',
        'num_states': 'log2(# States)'
    },
    opacity=0.2
).update_layout(
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.8,
        xanchor='left',
        x=1.2,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='rgba(0, 0, 0, 0.2)',
        borderwidth=1
    ),
    margin=dict(r=200),
    hovermode='closest',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=20)
)

for i, vocab_size in enumerate(sorted(df['num_symbols'].unique())):
    p, _ = curve_fit(
        f,
        df['entropy'][df['num_symbols'] == vocab_size],
        df['ee'][df['num_symbols'] == vocab_size],
        p0=[1,1,1,1]
    )

    x = np.linspace(
        df['entropy'][df['num_symbols'] == vocab_size].min(),
        df['entropy'][df['num_symbols'] == vocab_size].max(),
        1000
    )
    fig.add_trace(
        go.Scatter(
            x = x,
            y = f(x, p[0], p[1], p[2], p[3]),
            mode = 'lines',
            name = f'Fit (vocab_size={vocab_size})',
            line=dict(width=2)
        )
    )
fig.show()
fig.write_image('figs/excess_vs_entropy_pcfg.pdf', format='pdf', width=1000)

/home/drd92/pcfg-entropy/venv/lib64/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

invalid value encountered in log

/home/drd92/pcfg-entropy/venv/lib64/python3.11/site-packages/scipy/optimize/_minpack_py.py:493: RuntimeWarning:

overflow encountered in matmul

/home/drd92/pcfg-entropy/venv/lib64/python3.11/site-packages/scipy/optimize/_minpack_py.py:1041: RuntimeWarning:

invalid value encountered in multiply



In [14]:
fig = px.scatter(
    df,
    'entropy',
    'max_rho_num_steps',
    symbol='lstm_or_trf',
    size='num_symbols',
    title='Training steps to max SLAC@(1,10) vs. Entropy (size indicates # symbols)',
    color='num_states',
    labels={
        'entropy': 'Entropy (nats)',
        'max_rho_num_steps': 'Training steps to max SLAC@(1,10)',
        'lstm_or_trf': 'LSTM/TRF',
        'num_symbols': '# Symbols',
        'num_states': 'log_2(# States)'
    },
    opacity=0.5
).update_layout(
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.8,
        xanchor='left',
        x=1.2,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='rgba(0, 0, 0, 0.2)',
        borderwidth=1
    ),
    margin=dict(r=200),
    hovermode='closest',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=20)
)
fig.show()
fig.write_image('figs/slac_steps_pcfg.pdf', format='pdf', width=1000)

In [15]:
df['num_states'] = df['num_states'].astype(int)

model = smf.mixedlm(
    'max_rho_num_steps ~ num_symbols + num_states + lstm_or_trf + entropy',
    groups=df['seed'],
    data=df
)

result = model.fit()
print(result.summary())

                Mixed Linear Model Regression Results
Model:               MixedLM   Dependent Variable:   max_rho_num_steps
No. Observations:    52        Method:               REML             
No. Groups:          3         Scale:                11577156.5460    
Min. group size:     16        Log-Likelihood:       -466.2089        
Max. group size:     18        Converged:            Yes              
Mean group size:     17.3                                             
----------------------------------------------------------------------
                     Coef.    Std.Err.   z    P>|z|   [0.025   0.975] 
----------------------------------------------------------------------
Intercept            1313.258 2673.401  0.491 0.623 -3926.511 6553.028
lstm_or_trf[T.trf]   1576.923  943.690  1.671 0.095  -272.675 3426.521
num_symbols             0.112    0.253  0.441 0.659    -0.385    0.608
num_states           -869.004  637.458 -1.363 0.173 -2118.400  380.391
entropy               3